In [1]:
import json
import pickle
from pathlib import Path

import tensorflow as tf
from tensorflow.keras.layers import Dense, Embedding, Input, LSTM, SpatialDropout1D
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import tokenizer_from_json

# Use the CPU even if a GPU plugin is installed later.
# Run this cell before building a model; restart the kernel if devices were initialized.
tf.config.set_visible_devices([], "GPU")
tf.keras.utils.set_random_seed(42)
print("TensorFlow:", tf.__version__)
print("Training devices:", tf.config.get_visible_devices())

TensorFlow: 2.22.0-rc0
Training devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


## Local project paths
Find the project from either its root directory or the `nbs` directory.

In [ ]:
working_dir = Path.cwd().resolve()
PROJECT_DIR = next(
    (directory for directory in (working_dir, *working_dir.parents)
     if (directory / "nbs").is_dir() and (directory / "requirements.txt").is_file()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError("Open this notebook from the Spam-Detection-API project.")

EXPORT_DIR = PROJECT_DIR / "datasets" / "export"
MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
TRAINING_DATA_PATH = EXPORT_DIR / "spam-metadata.pkl"
TOKENIZER_INPUT_PATH = EXPORT_DIR / "spam-tokenizer.json"

for input_path in (TRAINING_DATA_PATH, TOKENIZER_INPUT_PATH):
    if not input_path.is_file():
        raise FileNotFoundError(f"Missing prepared data file: {input_path}")

print("Project:", PROJECT_DIR)
print("Prepared data:", EXPORT_DIR)
print("Model exports:", MODEL_DIR)

## Load prepared data
Load the pickle generated by your local preprocessing and reconstruct its matching tokenizer from JSON.

In [3]:
with TRAINING_DATA_PATH.open("rb") as file:
    data = pickle.load(file)

tokenizer = tokenizer_from_json(TOKENIZER_INPUT_PATH.read_text(encoding="utf-8"))
X_train = data["X_train"]
X_test = data["X_test"]
y_train = data["y_train"]
y_test = data["y_test"]
max_words = data["max_words"]
max_sequence = data["max_seq_length"]
legend = data["label_legend"]
labels_legend_inverted = data["label_legend_inverted"]

assert X_train.shape[1] == X_test.shape[1] == max_sequence
assert len(X_train) == len(y_train) and len(X_test) == len(y_test)
assert y_train.shape[1] == y_test.shape[1] == len(legend)
assert tokenizer.num_words == max_words
assert 0 <= X_train.min() and X_train.max() < max_words
assert 0 <= X_test.min() and X_test.max() < max_words
print(f"Training: {len(X_train):,} messages; test: {len(X_test):,} messages")

Training: 5,043 messages; test: 2,485 messages


## Build the LSTM model

In [4]:
embed_dim = 128
lstm_out = 196

model = Sequential([
    Input(shape=(max_sequence,), dtype="int32"),
    Embedding(input_dim=max_words, output_dim=embed_dim),
    SpatialDropout1D(0.4),
    LSTM(lstm_out, dropout=0.3, recurrent_dropout=0.0),
    Dense(len(legend), activation="softmax"),
])
model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 300, 128)       │        35,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ (None, 300, 128)       │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 196)            │       254,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │           394 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 291,034 (1.11 MB)

 Trainable params: 291,034 (1.11 MB)

 Non-trainable params: 0 (0.00 B)

## Train on the CPU

Start with one epoch if you want to measure runtime before a longer run. The existing test split is used for validation, matching the original notebook. If you use these scores to tune the model, reserve another untouched test set for final evaluation.

In [5]:
batch_size = 32
epochs = 5
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    batch_size=batch_size,
    epochs=epochs,
    verbose=1,
)

Epoch 1/5
158/158 ━━━━━━━━━━━━━━━━━━━━ 72s 450ms/step - accuracy: 0.8741 - loss: 0.3102 - val_accuracy: 0.9308 - val_loss: 0.1950
Epoch 2/5
158/158 ━━━━━━━━━━━━━━━━━━━━ 74s 472ms/step - accuracy: 0.9445 - loss: 0.1589 - val_accuracy: 0.9561 - val_loss: 0.1360
Epoch 3/5
158/158 ━━━━━━━━━━━━━━━━━━━━ 72s 456ms/step - accuracy: 0.9568 - loss: 0.1309 - val_accuracy: 0.9581 - val_loss: 0.1330
Epoch 4/5
158/158 ━━━━━━━━━━━━━━━━━━━━ 70s 442ms/step - accuracy: 0.9639 - loss: 0.1197 - val_accuracy: 0.9590 - val_loss: 0.1305
Epoch 5/5
158/158 ━━━━━━━━━━━━━━━━━━━━ 67s 425ms/step - accuracy: 0.9619 - loss: 0.1163 - val_accuracy: 0.9602 - val_loss: 0.1332


## Save the model, tokenizer, and metadata
Run after training. Rerunning this cell replaces the previous model exports. The prepared input files are kept separately.

In [ ]:
MODEL_EXPORT_PATH = MODEL_DIR / "spam-model.keras"
TOKENIZER_EXPORT_PATH = MODEL_DIR / "spam-classifier-tokenizer.json"
METADATA_EXPORT_PATH = MODEL_DIR / "spam-classifier-metadata.json"

metadata = {
    "preprocessing": "keras_legacy_tokenizer",
    "tensorflow_version": tf.__version__,
    "labels_legend_inverted": labels_legend_inverted,
    "legend": legend,
    "max_sequence": max_sequence,
    "max_words": max_words,
}
model.save(str(MODEL_EXPORT_PATH))
TOKENIZER_EXPORT_PATH.write_text(tokenizer.to_json(), encoding="utf-8")
METADATA_EXPORT_PATH.write_text(json.dumps(metadata, indent=4), encoding="utf-8")

for export_path in (MODEL_EXPORT_PATH, TOKENIZER_EXPORT_PATH, METADATA_EXPORT_PATH):
    print(f"Saved: {export_path}")

## Predict a new message

In [7]:
def predict(text):
    sequences = tokenizer.texts_to_sequences([text])
    x_input = pad_sequences(sequences, maxlen=max_sequence)
    probabilities = model.predict(x_input, verbose=0)[0]
    return {
        labels_legend_inverted[str(i)]: float(probability)
        for i, probability in enumerate(probabilities)
    }

predict("Hello world")

{'ham': 0.9858578443527222, 'spam': 0.014142151921987534}